# Exploração: 12 anos de histórico Spotify

Análise exploratória sobre as tabelas processadas em `data/processed/`.
Ver `reports/README.md` para o relatório final com storytelling.

In [ ]:
import pandas as pd

fact = pd.read_csv('../data/processed/fact_streams.csv', parse_dates=['date_key'])
artist = pd.read_csv('../data/processed/dim_artist.csv')
track = pd.read_csv('../data/processed/dim_track.csv')
fact = fact.merge(artist[['artist_id', 'artist_name']], on='artist_id')
fact['year'] = fact['date_key'].dt.year
fact['hours'] = fact['ms_played'] / 3_600_000
fact.head()

## Artista mais ouvido por ano (horas)

In [ ]:
top_per_year = fact.groupby(['year', 'artist_name'])['hours'].sum().reset_index()
idx = top_per_year.groupby('year')['hours'].idxmax()
top_per_year.loc[idx]

## Diversidade de artistas por ano

In [ ]:
fact.groupby('year')['artist_name'].nunique()

## Taxa de skip por ano (limitação: campo não confiável antes de 2022)

In [ ]:
fact.groupby('year')['skipped'].mean()

## Detecção do outlier de 17/07/2018

In [ ]:
daily = fact.groupby('date_key').agg(plays=('stream_id', 'count'), hours=('hours', 'sum'))
daily.sort_values('plays', ascending=False).head(10)